# Intake phase pipeline

In [ ]:
import os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

NEW_CODE = Path.cwd() if (Path.cwd() / "functions.py").exists() else Path.cwd() / "new_code"
sys.path[:0] = [str(NEW_CODE), str(NEW_CODE.parent)]
import functions as f

## 1. Load one subject

`X` is (n_epochs, n_times), one row per stimulus.

In [ ]:
# >>> UPDATE: folder containing <subject>/EEG/processed/<subject>_intake_stim-epo.fif <<<
DATA_ROOT = Path("/Volumes/CENSORLAB$/Or/Cortical Communication/CorticalCommunication/Data")
SUBJECT = "sub_102"   # sub_102-sub_113 (no sub_106)
DEMO = False or os.environ.get("PHASE_NOTEBOOK_DEMO") == "1"   # True = synthetic epochs, no data needed

In [ ]:
# Preprocessing defaults (Fz Hjorth channel, 100 Hz low-pass, 1000 Hz) live in f.LoadConfig;
# override a field here if needed, e.g. f.LoadConfig(data_root=DATA_ROOT, channel="Fz").
config = f.LoadConfig(data_root=DATA_ROOT)
data = f.make_synthetic_epochs(subject=SUBJECT, n_epochs=40, seed=0) if DEMO else f.get_data(SUBJECT, config)
print(f"{SUBJECT}: X {data.X.shape}, fs {data.fs:g} Hz")
pd.Series(data.labels).value_counts()

## 2. One epoch, step by step

Non-causal = offline phase using the whole epoch (reference). Causal = phase from pre-stimulus data only, as a real-time system would see it.

In [ ]:
EPOCH = 0                 # epoch to show
BAND = (4.0, 8.0)         # Hz, band whose phase is estimated (theta)
CUTOFF_MS = 0.0           # ms, stimulus time = where phase is read
FILTER_ORDER = 350        # FIR order of the non-causal filter (longer = sharper band edges)
CAUSAL_PARAMS = {
    "window_ms": 510.0,   # ms of pre-stimulus data the causal method may use
    "filter_order": FILTER_ORDER,
    "edge": 65,           # samples dropped at both window ends (filter transients)
    "ar_order": 78,       # autoregressive model order used to forecast past t=0
    "hilbert_window": 128,  # samples in the Hilbert transform
    "offset": 0,          # phase-index correction, samples
}

In [ ]:
x = data.X[EPOCH]
nc = f.noncausal_phase(x, data.times_ms, data.fs, BAND, FILTER_ORDER, CUTOFF_MS)
c = f.causal_phase(x, data.times_ms, data.fs, BAND, CUTOFF_MS, CAUSAL_PARAMS)
f.plot_epoch_estimates(x, data.times_ms, nc, c, BAND, CUTOFF_MS, title=f"{SUBJECT}, epoch {EPOCH} (BOSS {data.labels[EPOCH]})")
plt.show()

## 3. Optional: optimize the causal parameters for this subject

Tries `n_opt_trials` parameter sets and keeps the one whose causal phase best matches the non-causal phase (on 80 % of epochs; the held-out 20 % is printed). When optimized, the non-causal filter uses the optimized `filter_order` too, so both methods share one band-pass.

In [ ]:
RUN_OPTIMIZATION = False   # True takes about a minute
OPTIMIZATION_CONFIG = {
    "n_opt_trials": 200,       # parameter sets tried
    "train_fraction": 0.8,     # epochs used for fitting; the rest are held out
    "random_seed": 0,
    "min_usable_epochs": 20,
    "window_ms_range": (300, 950), "window_ms_step": 10,
    "filter_order_range": (150, 450), "filter_order_step": 5,
    "edge_range": (20, 200), "edge_step": 5,
    "ar_order_range": (5, 80),
    "hilbert_window": 128, "offset": 0, "infeasible_penalty": 10.0,
}

In [ ]:
if RUN_OPTIMIZATION:
    params = f.optimize_causal_params(data.X, data.times_ms, data.fs, BAND, CUTOFF_MS, OPTIMIZATION_CONFIG, name=SUBJECT)
else:
    params = f.validate_causal_params(CAUSAL_PARAMS)
noncausal_filter_order = f.resolve_noncausal_filter_order(params, FILTER_ORDER)
params

## 4. All epochs: BOSS success

**Success** = % of BOSS-labeled epochs whose non-causal phase is within ±T of the BOSS target (0° positive, 180° negative); epochs outside both windows (unclassified) count as misses. **Chance** = 2T/360, the success expected if BOSS fired at a random phase.

In [ ]:
TOLERANCES_DEG = (45, 30, 15)   # +/- deg windows around the BOSS target

In [ ]:
estimates = f.estimate_subject_epochs(data, BAND, noncausal_filter_order, CUTOFF_MS, n_epochs=False, causal_params=params)
arrays = f.estimates_to_arrays(estimates)
analysis = f.analyze_phase_results({SUBJECT: arrays}, TOLERANCES_DEG)
rows = []
for t in TOLERANCES_DEG:
    c = analysis["subjects"][SUBJECT]["counts"]["noncausal"][str(t)]
    pos, neg = c["by_class"][f.POSITIVE], c["by_class"][f.NEGATIVE]
    rows.append({"tolerance": f"+/-{t}",
                 "BOSS positive": pos["n"], "-> non-causal positive": pos["correct"],
                 "BOSS negative": neg["n"], "-> non-causal negative": neg["correct"],
                 "total": c["n_labeled"], "total correct": c["correct"],
                 "success %": round(c["success_pct"], 1), "chance %": round(f.chance_success_pct(t), 1)})
pd.DataFrame(rows).set_index("tolerance")

## 5. Figures

Phase circles: each epoch's phase at t=0, colored by BOSS label. Circular histograms (rose plots): epochs per 10° bin, stacked by BOSS label; arrows = mean direction and concentration R per BOSS label. First the non-causal phase itself, then the deviations from the BOSS target and between the two methods.

In [ ]:
TOLERANCE_DEG = 30   # window shaded in the circular plots

In [ ]:
histograms = analysis["subjects"][SUBJECT]["histograms"]
fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw={"projection": "polar"})
f.plot_phase_circle(arrays["noncausal_deg"], arrays["boss"], TOLERANCE_DEG, title="Non-causal phase", ax=axes[0, 0])
f.plot_phase_circle(arrays["causal_deg"], arrays["boss"], TOLERANCE_DEG, title="Causal phase", ax=axes[0, 1])
f.plot_circular_histogram(histograms["noncausal_phase"], TOLERANCE_DEG, title="Non-causal phase histogram", ax=axes[0, 2])
f.plot_circular_histogram(histograms["boss_target"], TOLERANCE_DEG, title="Phase - BOSS target", ax=axes[1, 0])
f.plot_circular_histogram(histograms["causal_error"], TOLERANCE_DEG, title="Causal - non-causal", ax=axes[1, 1])
axes[1, 2].remove()
f.plot_success_vs_tolerance(analysis["success"])
plt.show()